## 빈 components 레코드 제거

`components`가 빈 리스트인 레코드는 재료 정보가 없으므로 제거한다.

In [1]:
from pathlib import Path
import json

GRAPH_INPUT_PATH = next(Path(".").glob("*/recipes_graph_prepared_v2.jsonl"))
GRAPH_OUTPUT_PATH = GRAPH_INPUT_PATH.with_name("recipes_graph_prepared_v2_nonempty.jsonl")

input_records = []
removed_records = 0
with GRAPH_INPUT_PATH.open(encoding="utf-8") as input_file:
    for line_number, line in enumerate(input_file, start=1):
        if not line.strip():
            continue
        try:
            record = json.loads(line)
        except json.JSONDecodeError as error:
            raise ValueError(f"JSON 파싱 실패: {line_number}번째 줄") from error

        if record.get("components") == []:
            removed_records += 1
            continue
        input_records.append(record)

GRAPH_OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
with GRAPH_OUTPUT_PATH.open("w", encoding="utf-8") as output_file:
    for record in input_records:
        output_file.write(json.dumps(record, ensure_ascii=False) + "\n")

print(f"원본 레코드 수: {len(input_records) + removed_records:,}")
print(f"삭제된 레코드 수: {removed_records:,}")
print(f"남은 레코드 수: {len(input_records):,}")
print(f"저장 경로: {GRAPH_OUTPUT_PATH}")

원본 레코드 수: 9,985
삭제된 레코드 수: 430
남은 레코드 수: 9,555
저장 경로: 전처리\recipes_graph_prepared_v2_nonempty.jsonl


In [2]:
# 결과에 components가 빈 리스트인 레코드가 남아 있지 않은지 검증한다.
with GRAPH_OUTPUT_PATH.open(encoding="utf-8") as output_file:
    saved_records = [json.loads(line) for line in output_file if line.strip()]

assert len(saved_records) == len(input_records)
assert all(record.get("components") != [] for record in saved_records)
print("검증 완료: 빈 components 레코드가 제거되었습니다.")

검증 완료: 빈 components 레코드가 제거되었습니다.
